Promotes validated Databricks analytics outputs into governed Fabric Gold tables for supplier risk, pricing anomalies, and savings opportunities, with Gold surrogate keys, lineage metadata, quality validation, and historical prediction snapshots.

**Imports and configuration**

In [0]:
# ============================================================
# DB_07_Score_and_Write_ML_Outputs
#
# Purpose:
# - Read validated outputs from DB_03, DB_05 and DB_06
# - Enrich ML outputs with Gold surrogate keys
# - Validate output grain and referential integrity
# - Publish governed ML data products into Fabric Gold
#
# IMPORTANT:
# This notebook does NOT retrain or rescore models.
# Scoring has already been completed upstream.
# ============================================================

from datetime import date, datetime, timezone

import uuid

from delta.tables import DeltaTable

from pyspark.sql import functions as F
from pyspark.sql import types as T


# ------------------------------------------------------------
# Business prediction snapshot
# ------------------------------------------------------------

PREDICTION_DATE = date(
    2026,
    7,
    31
)

PREDICTION_DATE_KEY = 20260731


# ------------------------------------------------------------
# Promotion metadata
# ------------------------------------------------------------

PROMOTION_BATCH_ID = str(
    uuid.uuid4()
)

PROMOTION_TIMESTAMP_UTC = datetime.now(
    timezone.utc
)

PROMOTION_LOAD_DATE = (
    PROMOTION_TIMESTAMP_UTC.date()
)


# ------------------------------------------------------------
# Source notebooks
# ------------------------------------------------------------

SUPPLIER_RISK_SOURCE_NOTEBOOK = (
    "DB_03_Train_Supplier_Risk_Model"
)

PRICING_ANOMALY_SOURCE_NOTEBOOK = (
    "DB_05_Train_Pricing_Anomaly_Model"
)

SAVINGS_OPPORTUNITY_SOURCE_NOTEBOOK = (
    "DB_06_Build_Savings_Opportunity_Engine"
)


# ------------------------------------------------------------
# Ensure deterministic timestamp handling
# ------------------------------------------------------------

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


print(
    "DB_07 configuration loaded."
)

print(
    "Prediction date:",
    PREDICTION_DATE
)

print(
    "Prediction date key:",
    PREDICTION_DATE_KEY
)

print(
    "Promotion batch:",
    PROMOTION_BATCH_ID
)

print(
    "Promotion timestamp UTC:",
    PROMOTION_TIMESTAMP_UTC
)

DB_07 configuration loaded.
Prediction date: 2026-07-31
Prediction date key: 20260731
Promotion batch: a222cd41-aa4e-4afa-848d-ccb194f35f4a
Promotion timestamp UTC: 2026-08-13 15:23:04.248548+00:00


**Load OneLake credentials**

In [0]:
# ============================================================
# Load Fabric OneLake credentials securely
# ============================================================

tenant_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-id"
)

client_secret = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-secret"
)


print(
    "Fabric OneLake credentials loaded securely."
)

Fabric OneLake credentials loaded securely.


**Configure OneLake OAuth**

In [0]:
# ============================================================
# Configure Fabric OneLake OAuth
# ============================================================

spark.conf.set(
    "fs.azure.account.auth.type",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint",
    (
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )
)


print(
    "OneLake OAuth configuration applied."
)

OneLake OAuth configuration applied.


**Define source and Gold target paths**

In [0]:
# ============================================================
# Source and target paths
# ============================================================

GOLD_LAKEHOUSE_ROOT = (
    "<ABFSS PATH>"
    "<LAKEHOUSE ID>"
)


# ============================================================
# DB_03 source
# ============================================================

SUPPLIER_RISK_SOURCE_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/supplier_risk/"
    f"predictions_2026"
)


# ============================================================
# DB_05 source
# ============================================================

PRICING_ANOMALY_SOURCE_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/pricing_anomaly/"
    f"scoring_predictions"
)


# ============================================================
# DB_06 source
# ============================================================

SAVINGS_OPPORTUNITY_SOURCE_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/savings_opportunity/"
    f"opportunities_2026"
)


# ============================================================
# Existing Gold reference data
# ============================================================

DIM_SUPPLIER_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_supplier"
)

DIM_CATEGORY_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_category"
)

DIM_DATE_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_date"
)

FACT_PURCHASE_ORDER_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/fact_purchase_order"
)


# ============================================================
# Physical Gold ML outputs
# ============================================================

ML_SUPPLIER_RISK_TARGET_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/ml_supplier_risk_prediction"
)

ML_PRICING_ANOMALY_TARGET_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/ml_pricing_anomaly_prediction"
)

ML_SAVINGS_OPPORTUNITY_TARGET_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/ml_savings_opportunity"
)


# ============================================================
# Promotion monitoring
# ============================================================

ML_PROMOTION_MONITORING_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/monitoring_ml_gold_promotion_results"
)


print(
    "DB_07 paths configured."
)

print(
    "\nSupplier risk target:",
    ML_SUPPLIER_RISK_TARGET_PATH
)

print(
    "Pricing anomaly target:",
    ML_PRICING_ANOMALY_TARGET_PATH
)

print(
    "Savings opportunity target:",
    ML_SAVINGS_OPPORTUNITY_TARGET_PATH
)

**Utility functions**

In [0]:
# ============================================================
# DB_07 utility functions
# ============================================================

def require_columns(
    dataframe,
    required_columns,
    dataframe_name
):

    missing_columns = [
        column_name
        for column_name in required_columns
        if column_name not in dataframe.columns
    ]


    if missing_columns:

        raise ValueError(
            f"{dataframe_name} is missing required columns: "
            +
            ", ".join(
                missing_columns
            )
        )


    print(
        f"{dataframe_name} schema contract PASSED."
    )


def qualified_optional_column(
    dataframe,
    dataframe_alias,
    column_name,
    data_type,
    output_name=None
):

    output_name = (
        output_name
        if output_name is not None
        else column_name
    )


    if column_name in dataframe.columns:

        return (
            F.col(
                f"{dataframe_alias}.{column_name}"
            )
            .cast(
                data_type
            )
            .alias(
                output_name
            )
        )


    return (
        F.lit(
            None
        )
        .cast(
            data_type
        )
        .alias(
            output_name
        )
    )


def add_gold_ml_lineage(
    dataframe,
    source_notebook,
    source_path
):

    return (
        dataframe

        .withColumn(
            "PromotionBatchID",
            F.lit(
                PROMOTION_BATCH_ID
            )
        )

        .withColumn(
            "MLSourceNotebook",
            F.lit(
                source_notebook
            )
        )

        .withColumn(
            "MLSourcePath",
            F.lit(
                source_path
            )
        )

        .withColumn(
            "GoldMLLoadTimestampUTC",
            F.lit(
                PROMOTION_TIMESTAMP_UTC
            )
            .cast(
                "timestamp"
            )
        )

        .withColumn(
            "GoldMLLoadDate",
            F.lit(
                PROMOTION_LOAD_DATE
            )
            .cast(
                "date"
            )
        )
    )


def write_prediction_snapshot(
    dataframe,
    target_path,
    prediction_date_column="PredictionDate"
):

    snapshot_predicate = (
        f"{prediction_date_column} = "
        f"DATE '{PREDICTION_DATE.isoformat()}'"
    )


    if DeltaTable.isDeltaTable(
        spark,
        target_path
    ):

        (
            dataframe

            .write

            .format(
                "delta"
            )

            .mode(
                "overwrite"
            )

            .option(
                "replaceWhere",
                snapshot_predicate
            )

            .save(
                target_path
            )
        )


        print(
            "Replaced prediction snapshot:",
            target_path
        )


    else:

        (
            dataframe

            .write

            .format(
                "delta"
            )

            .mode(
                "overwrite"
            )

            .option(
                "overwriteSchema",
                "true"
            )

            .save(
                target_path
            )
        )


        print(
            "Created prediction table:",
            target_path
        )


print(
    "DB_07 utility functions loaded."
)

DB_07 utility functions loaded.


**Read validated ML outputs and Gold references**

In [0]:
# ============================================================
# Read DB_03 / DB_05 / DB_06 outputs
# ============================================================

supplier_risk_source_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        SUPPLIER_RISK_SOURCE_PATH
    )
)


pricing_anomaly_source_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        PRICING_ANOMALY_SOURCE_PATH
    )
)


savings_opportunity_source_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        SAVINGS_OPPORTUNITY_SOURCE_PATH
    )
)


# ============================================================
# Gold references
# ============================================================

dim_supplier_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        DIM_SUPPLIER_PATH
    )
)


dim_category_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        DIM_CATEGORY_PATH
    )
)


dim_date_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        DIM_DATE_PATH
    )
)


fact_po_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        FACT_PURCHASE_ORDER_PATH
    )
)


# ============================================================
# Counts
# ============================================================

supplier_risk_source_count = (
    supplier_risk_source_df.count()
)

pricing_anomaly_source_count = (
    pricing_anomaly_source_df.count()
)

savings_opportunity_source_count = (
    savings_opportunity_source_df.count()
)


print(
    "Supplier risk predictions:",
    f"{supplier_risk_source_count:,}"
)

print(
    "Pricing anomaly predictions:",
    f"{pricing_anomaly_source_count:,}"
)

print(
    "Savings opportunities:",
    f"{savings_opportunity_source_count:,}"
)

Supplier risk predictions: 356
Pricing anomaly predictions: 21,752
Savings opportunities: 983


**Validate source contracts**

In [0]:
# ============================================================
# Validate upstream ML source contracts
# ============================================================

require_columns(
    supplier_risk_source_df,
    [
        "SupplierID",
        "SupplierRiskScore",
        "PredictedHighRiskFlag",
        "ModelName",
        "ModelRunID"
    ],
    "DB_03 supplier risk output"
)


require_columns(
    pricing_anomaly_source_df,
    [
        "POItemID",
        "PricingAnomalyScore",
        "PricingAnomalyFlag",
        "ModelName",
        "ModelRunID"
    ],
    "DB_05 pricing anomaly output"
)


require_columns(
    savings_opportunity_source_df,
    [
        "SupplierID",
        "CategoryID",
        "PredictionDate",
        "PotentialAnnualSavingsEUR",
        "NegotiationPriorityScore",
        "NegotiationPriority",
        "SavingsOpportunityRank",
        "EngineRunID"
    ],
    "DB_06 savings opportunity output"
)


require_columns(
    fact_po_df,
    [
        "POItemID",
        "PurchaseOrderFactKey",
        "POID",
        "OrderDate",
        "SupplierKey",
        "MaterialKey",
        "CategoryKey",
        "ContractKey",
        "SupplierID",
        "MaterialID",
        "CategoryID",
        "ContractID"
    ],
    "Gold fact_purchase_order"
)


print(
    "\nAll DB_07 source contracts PASSED."
)

DB_03 supplier risk output schema contract PASSED.
DB_05 pricing anomaly output schema contract PASSED.
DB_06 savings opportunity output schema contract PASSED.
Gold fact_purchase_order schema contract PASSED.

All DB_07 source contracts PASSED.


**Build Gold reference key maps**

In [0]:
# ============================================================
# Build Gold dimension / fact key maps
# ============================================================

prediction_date_literal = (
    F.lit(
        PREDICTION_DATE
    )
    .cast(
        "date"
    )
)


# ============================================================
# Supplier SCD2 mapping valid AS OF PredictionDate
#
# Do not use IsCurrentFlag here.
#
# Historical prediction snapshots must resolve to the Supplier
# dimension version that was valid on the prediction date.
# ============================================================

supplier_as_of_map_df = (
    dim_supplier_df

    .filter(
        (
            F.to_date(
                F.col(
                    "EffectiveFromDate"
                )
            )
            <=
            prediction_date_literal
        )
        &
        (
            F.coalesce(
                F.to_date(
                    F.col(
                        "EffectiveToDate"
                    )
                ),

                F.lit(
                    "9999-12-31"
                )
                .cast(
                    "date"
                )
            )
            >=
            prediction_date_literal
        )
    )

    .select(
        F.col(
            "SupplierID"
        )
        .cast(
            "string"
        )
        .alias(
            "SupplierID"
        ),

        F.col(
            "SupplierKey"
        )
        .cast(
            "long"
        )
        .alias(
            "SupplierKey"
        ),

        F.col(
            "DimensionVersion"
        )
        .cast(
            "int"
        )
        .alias(
            "SupplierDimensionVersion"
        ),

        F.to_date(
            F.col(
                "EffectiveFromDate"
            )
        )
        .alias(
            "SupplierEffectiveFromDate"
        ),

        F.to_date(
            F.col(
                "EffectiveToDate"
            )
        )
        .alias(
            "SupplierEffectiveToDate"
        )
    )
)


supplier_as_of_count = (
    supplier_as_of_map_df.count()
)


supplier_as_of_duplicate_count = (
    supplier_as_of_map_df

    .groupBy(
        "SupplierID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


supplier_as_of_null_key_count = (
    supplier_as_of_map_df

    .filter(
        F.col(
            "SupplierID"
        ).isNull()
        |
        F.col(
            "SupplierKey"
        ).isNull()
    )

    .count()
)


if supplier_as_of_count == 0:

    raise ValueError(
        "Supplier SCD2 mapping contains no records "
        "valid on PredictionDate."
    )


if supplier_as_of_duplicate_count > 0:

    raise ValueError(
        "Supplier SCD2 mapping contains multiple dimension "
        "versions valid for the same SupplierID on "
        f"{PREDICTION_DATE}. "
        f"Duplicate suppliers: {supplier_as_of_duplicate_count}"
    )


if supplier_as_of_null_key_count > 0:

    raise ValueError(
        "Supplier SCD2 mapping contains null "
        "SupplierID or SupplierKey values."
    )


# ============================================================
# Category Type 1 mapping
#
# Do NOT silently drop duplicates.
# Gold should already satisfy one CategoryID -> one CategoryKey.
# ============================================================

category_map_df = (
    dim_category_df

    .select(
        F.col(
            "CategoryID"
        )
        .cast(
            "string"
        )
        .alias(
            "CategoryID"
        ),

        F.col(
            "CategoryKey"
        )
        .cast(
            "long"
        )
        .alias(
            "CategoryKey"
        )
    )
)


category_duplicate_count = (
    category_map_df

    .groupBy(
        "CategoryID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


category_null_key_count = (
    category_map_df

    .filter(
        F.col(
            "CategoryID"
        ).isNull()
        |
        F.col(
            "CategoryKey"
        ).isNull()
    )

    .count()
)


category_map_count = (
    category_map_df.count()
)


if category_map_count == 0:

    raise ValueError(
        "Category mapping contains no rows."
    )


if category_duplicate_count > 0:

    raise ValueError(
        "dim_category violates expected Type 1 grain. "
        f"Duplicate CategoryID count: "
        f"{category_duplicate_count}"
    )


if category_null_key_count > 0:

    raise ValueError(
        "Category mapping contains null "
        "CategoryID or CategoryKey values."
    )


# ============================================================
# Purchase Order fact key mapping
#
# POItemID must already be unique in fact_purchase_order.
# Do NOT hide duplicates with dropDuplicates().
# ============================================================

po_key_map_df = (
    fact_po_df

    .select(
        F.col(
            "POItemID"
        )
        .cast(
            "string"
        )
        .alias(
            "POItemID"
        ),

        F.col(
            "PurchaseOrderFactKey"
        )
        .cast(
            "long"
        )
        .alias(
            "PurchaseOrderFactKey"
        ),

        F.col(
            "POID"
        )
        .cast(
            "string"
        )
        .alias(
            "POID"
        ),

        F.to_date(
            F.col(
                "OrderDate"
            )
        )
        .alias(
            "OrderDate"
        ),

        F.col(
            "SupplierKey"
        )
        .cast(
            "long"
        )
        .alias(
            "SupplierKey"
        ),

        F.col(
            "MaterialKey"
        )
        .cast(
            "long"
        )
        .alias(
            "MaterialKey"
        ),

        F.col(
            "CategoryKey"
        )
        .cast(
            "long"
        )
        .alias(
            "CategoryKey"
        ),

        F.col(
            "ContractKey"
        )
        .cast(
            "long"
        )
        .alias(
            "ContractKey"
        ),

        F.col(
            "SupplierID"
        )
        .cast(
            "string"
        )
        .alias(
            "SupplierID"
        ),

        F.col(
            "MaterialID"
        )
        .cast(
            "string"
        )
        .alias(
            "MaterialID"
        ),

        F.col(
            "CategoryID"
        )
        .cast(
            "string"
        )
        .alias(
            "CategoryID"
        ),

        F.col(
            "ContractID"
        )
        .cast(
            "string"
        )
        .alias(
            "ContractID"
        )
    )
)


po_duplicate_count = (
    po_key_map_df

    .groupBy(
        "POItemID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


po_null_fact_key_count = (
    po_key_map_df

    .filter(
        F.col(
            "POItemID"
        ).isNull()
        |
        F.col(
            "PurchaseOrderFactKey"
        ).isNull()
    )

    .count()
)


po_key_map_count = (
    po_key_map_df.count()
)


if po_key_map_count == 0:

    raise ValueError(
        "Purchase Order key mapping contains no rows."
    )


if po_duplicate_count > 0:

    raise ValueError(
        "fact_purchase_order violates POItemID grain. "
        f"Duplicate POItemID count: {po_duplicate_count}"
    )


if po_null_fact_key_count > 0:

    raise ValueError(
        "Purchase Order mapping contains null "
        "POItemID or PurchaseOrderFactKey values."
    )


# ============================================================
# Prediction Date dimension validation
# ============================================================

prediction_date_dimension_count = (
    dim_date_df

    .filter(
        F.col(
            "DateKey"
        )
        ==
        F.lit(
            PREDICTION_DATE_KEY
        )
    )

    .count()
)


if prediction_date_dimension_count != 1:

    raise ValueError(
        "PredictionDateKey does not resolve to exactly "
        "one dim_date record. "
        f"DateKey={PREDICTION_DATE_KEY}; "
        f"row count={prediction_date_dimension_count}"
    )


# ============================================================
# Diagnostics
# ============================================================

print(
    "Gold reference key maps prepared."
)

print(
    "\nSupplier SCD2 records valid on prediction date:",
    f"{supplier_as_of_count:,}"
)

print(
    "Supplier SCD2 duplicate mappings:",
    supplier_as_of_duplicate_count
)

print(
    "Category mappings:",
    f"{category_map_count:,}"
)

print(
    "Category duplicate mappings:",
    category_duplicate_count
)

print(
    "PO-item mappings:",
    f"{po_key_map_count:,}"
)

print(
    "PO-item duplicate mappings:",
    po_duplicate_count
)

print(
    "Prediction DateKey:",
    PREDICTION_DATE_KEY
)

print(
    "\nGold reference key-map validation PASSED."
)

Gold reference key maps prepared.

Supplier SCD2 records valid on prediction date: 500
Supplier SCD2 duplicate mappings: 0
Category mappings: 20
Category duplicate mappings: 0
PO-item mappings: 75,994
PO-item duplicate mappings: 0
Prediction DateKey: 20260731

Gold reference key-map validation PASSED.


**Build ml_supplier_risk_prediction**

In [0]:
# ============================================================
# Build Gold Supplier Risk Prediction
#
# Grain:
# SupplierID × PredictionDate
# ============================================================

supplier_risk_gold_df = (
    supplier_risk_source_df.alias(
        "sr"
    )

    .join(
        supplier_as_of_map_df.alias(
            "ds"
        ),

        F.col(
            "sr.SupplierID"
        )
        ==
        F.col(
            "ds.SupplierID"
        ),

        "left"
    )

    .select(
        F.lit(
            PREDICTION_DATE
        )
        .cast(
            "date"
        )
        .alias(
            "PredictionDate"
        ),

        F.lit(
            PREDICTION_DATE_KEY
        )
        .cast(
            "int"
        )
        .alias(
            "PredictionDateKey"
        ),

        F.col(
            "ds.SupplierKey"
        )
        .cast(
            "long"
        )
        .alias(
            "SupplierKey"
        ),

        F.col(
            "ds.SupplierDimensionVersion"
        )
        .cast(
            "int"
        )
        .alias(
            "SupplierDimensionVersion"
        ),

        F.col(
            "sr.SupplierID"
        )
        .cast(
            "string"
        )
        .alias(
            "SupplierID"
        ),

        F.col(
            "sr.SupplierRiskScore"
        )
        .cast(
            "double"
        )
        .alias(
            "SupplierRiskScore"
        ),

        F.col(
            "sr.PredictedHighRiskFlag"
        )
        .cast(
            "int"
        )
        .alias(
            "PredictedHighRiskFlag"
        ),

        F.col(
            "sr.ModelName"
        )
        .cast(
            "string"
        )
        .alias(
            "ModelName"
        ),

        F.col(
            "sr.ModelRunID"
        )
        .cast(
            "string"
        )
        .alias(
            "ModelRunID"
        ),

        qualified_optional_column(
            supplier_risk_source_df,
            "sr",
            "ModelStatus",
            "string"
        ),

        qualified_optional_column(
            supplier_risk_source_df,
            "sr",
            "DecisionThreshold",
            "double"
        ),

        qualified_optional_column(
            supplier_risk_source_df,
            "sr",
            "PredictionTimestampUTC",
            "timestamp"
        )
    )


    # --------------------------------------------------------
    # Deterministic key follows the declared business grain:
    # SupplierID × PredictionDate
    #
    # ModelRunID is lineage, not part of the business key.
    # --------------------------------------------------------

    .withColumn(
        "SupplierRiskPredictionKey",

        F.xxhash64(
            F.col(
                "SupplierID"
            ),

            F.col(
                "PredictionDate"
            )
            .cast(
                "string"
            )
        )
    )
)


supplier_risk_gold_df = (
    add_gold_ml_lineage(
        supplier_risk_gold_df,
        SUPPLIER_RISK_SOURCE_NOTEBOOK,
        SUPPLIER_RISK_SOURCE_PATH
    )
)


print(
    "Gold Supplier Risk output built:",
    f"{supplier_risk_gold_df.count():,}"
)

Gold Supplier Risk output built: 356


**Build _ml_pricing_anomaly_prediction_**

In [0]:
# ============================================================
# Build Gold Pricing Anomaly Prediction
#
# Grain:
# POItemID × PredictionDate
# ============================================================

pricing_anomaly_gold_df = (
    pricing_anomaly_source_df.alias(
        "pa"
    )

    .join(
        po_key_map_df.alias(
            "po"
        ),

        F.col(
            "pa.POItemID"
        )
        ==
        F.col(
            "po.POItemID"
        ),

        "left"
    )

    .select(
        F.lit(
            PREDICTION_DATE
        )
        .cast(
            "date"
        )
        .alias(
            "PredictionDate"
        ),

        F.lit(
            PREDICTION_DATE_KEY
        )
        .cast(
            "int"
        )
        .alias(
            "PredictionDateKey"
        ),

        F.col(
            "po.PurchaseOrderFactKey"
        )
        .cast(
            "long"
        )
        .alias(
            "PurchaseOrderFactKey"
        ),

        F.col(
            "pa.POItemID"
        )
        .cast(
            "string"
        )
        .alias(
            "POItemID"
        ),

        F.col(
            "po.POID"
        )
        .cast(
            "string"
        )
        .alias(
            "POID"
        ),

        F.to_date(
            F.col(
                "po.OrderDate"
            )
        )
        .alias(
            "OrderDate"
        ),

        F.col(
            "po.SupplierKey"
        )
        .cast(
            "long"
        )
        .alias(
            "SupplierKey"
        ),

        F.col(
            "po.MaterialKey"
        )
        .cast(
            "long"
        )
        .alias(
            "MaterialKey"
        ),

        F.col(
            "po.CategoryKey"
        )
        .cast(
            "long"
        )
        .alias(
            "CategoryKey"
        ),

        F.col(
            "po.ContractKey"
        )
        .cast(
            "long"
        )
        .alias(
            "ContractKey"
        ),

        F.col(
            "po.SupplierID"
        )
        .cast(
            "string"
        )
        .alias(
            "SupplierID"
        ),

        F.col(
            "po.MaterialID"
        )
        .cast(
            "string"
        )
        .alias(
            "MaterialID"
        ),

        F.col(
            "po.CategoryID"
        )
        .cast(
            "string"
        )
        .alias(
            "CategoryID"
        ),

        F.col(
            "po.ContractID"
        )
        .cast(
            "string"
        )
        .alias(
            "ContractID"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "UnitPriceEUR",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "LineAmountEUR",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "RawAnomalyScore",
            "double"
        ),

        F.col(
            "pa.PricingAnomalyScore"
        )
        .cast(
            "double"
        )
        .alias(
            "PricingAnomalyScore"
        ),

        F.col(
            "pa.PricingAnomalyFlag"
        )
        .cast(
            "int"
        )
        .alias(
            "PricingAnomalyFlag"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "RuleBasedExtremePriceFlag",
            "int"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "PriceComplianceExceptionFlag",
            "int"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "ContractPriceVariancePct",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "PriceVsMaterialHistoricalAvgPct",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "PriceVsSupplierMaterialHistoricalAvgPct",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "PriceVsCategoryHistoricalAvgPct",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "BenchmarkCoverageCount",
            "int"
        ),

        F.col(
            "pa.ModelName"
        )
        .cast(
            "string"
        )
        .alias(
            "ModelName"
        ),

        F.col(
            "pa.ModelRunID"
        )
        .cast(
            "string"
        )
        .alias(
            "ModelRunID"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "ModelStage",
            "string"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "ModelStatus",
            "string"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "AnomalyReviewRate",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "RawAnomalyThreshold",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "AnomalyScoreCutoff",
            "double"
        ),

        qualified_optional_column(
            pricing_anomaly_source_df,
            "pa",
            "PredictionTimestampUTC",
            "timestamp"
        )
    )


    # --------------------------------------------------------
    # Deterministic key follows:
    # POItemID × PredictionDate
    # --------------------------------------------------------

    .withColumn(
        "PricingAnomalyPredictionKey",

        F.xxhash64(
            F.col(
                "POItemID"
            ),

            F.col(
                "PredictionDate"
            )
            .cast(
                "string"
            )
        )
    )
)


pricing_anomaly_gold_df = (
    add_gold_ml_lineage(
        pricing_anomaly_gold_df,
        PRICING_ANOMALY_SOURCE_NOTEBOOK,
        PRICING_ANOMALY_SOURCE_PATH
    )
)


print(
    "Gold Pricing Anomaly output built:",
    f"{pricing_anomaly_gold_df.count():,}"
)

Gold Pricing Anomaly output built: 21,752


**Build ml_savings_opportunity**

In [0]:
# ============================================================
# Build Gold Savings Opportunity
#
# Grain:
# SupplierID × CategoryID × PredictionDate
# ============================================================

savings_opportunity_gold_df = (
    savings_opportunity_source_df.alias(
        "so"
    )

    .join(
        supplier_as_of_map_df.alias(
            "ds"
        ),

        F.col(
            "so.SupplierID"
        )
        ==
        F.col(
            "ds.SupplierID"
        ),

        "left"
    )

    .join(
        category_map_df.alias(
            "dc"
        ),

        F.col(
            "so.CategoryID"
        )
        ==
        F.col(
            "dc.CategoryID"
        ),

        "left"
    )

    .select(
        F.to_date(
            F.col(
                "so.PredictionDate"
            )
        )
        .alias(
            "PredictionDate"
        ),

        F.lit(
            PREDICTION_DATE_KEY
        )
        .cast(
            "int"
        )
        .alias(
            "PredictionDateKey"
        ),

        F.col(
            "ds.SupplierKey"
        )
        .cast(
            "long"
        )
        .alias(
            "SupplierKey"
        ),

        F.col(
            "ds.SupplierDimensionVersion"
        )
        .cast(
            "int"
        )
        .alias(
            "SupplierDimensionVersion"
        ),

        F.col(
            "dc.CategoryKey"
        )
        .cast(
            "long"
        )
        .alias(
            "CategoryKey"
        ),

        F.col(
            "so.SupplierID"
        )
        .cast(
            "string"
        )
        .alias(
            "SupplierID"
        ),

        F.col(
            "so.CategoryID"
        )
        .cast(
            "string"
        )
        .alias(
            "CategoryID"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "EligibleSpendYTDEUR",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "AnnualizedEligibleSpendEUR",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "AnnualizedPricingOpportunityEUR",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "AnnualizedMaverickOpportunityEUR",
            "double"
        ),

        F.col(
            "so.PotentialAnnualSavingsEUR"
        )
        .cast(
            "double"
        )
        .alias(
            "PotentialAnnualSavingsEUR"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "PotentialSavingsPct",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "PrimaryOpportunityDriver",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "PricingAnomalyRatePct",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "MaverickSpendPct",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "SupplierCategorySpendSharePct",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "SupplierRiskScore",
            "double"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "PredictedHighRiskFlag",
            "int"
        ),

        F.col(
            "so.NegotiationPriorityScore"
        )
        .cast(
            "double"
        )
        .alias(
            "NegotiationPriorityScore"
        ),

        F.col(
            "so.NegotiationPriority"
        )
        .cast(
            "string"
        )
        .alias(
            "NegotiationPriority"
        ),

        F.col(
            "so.SavingsOpportunityRank"
        )
        .cast(
            "long"
        )
        .alias(
            "SavingsOpportunityRank"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "CategorySavingsRank",
            "long"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "NegotiationPriorityRank",
            "long"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "ActionableOpportunityFlag",
            "int"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "PricingModelName",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "PricingModelRunID",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "SupplierRiskModelName",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "SupplierRiskModelRunID",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "EngineName",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "EngineVersion",
            "string"
        ),

        qualified_optional_column(
            savings_opportunity_source_df,
            "so",
            "EngineStatus",
            "string"
        ),

        F.col(
            "so.EngineRunID"
        )
        .cast(
            "string"
        )
        .alias(
            "EngineRunID"
        )
    )


    # --------------------------------------------------------
    # Deterministic key follows:
    # SupplierID × CategoryID × PredictionDate
    #
    # EngineRunID remains lineage only.
    # --------------------------------------------------------

    .withColumn(
        "SavingsOpportunityKey",

        F.xxhash64(
            F.col(
                "SupplierID"
            ),

            F.col(
                "CategoryID"
            ),

            F.col(
                "PredictionDate"
            )
            .cast(
                "string"
            )
        )
    )
)


savings_opportunity_gold_df = (
    add_gold_ml_lineage(
        savings_opportunity_gold_df,
        SAVINGS_OPPORTUNITY_SOURCE_NOTEBOOK,
        SAVINGS_OPPORTUNITY_SOURCE_PATH
    )
)


print(
    "Gold Savings Opportunity output built:",
    f"{savings_opportunity_gold_df.count():,}"
)

Gold Savings Opportunity output built: 983


**Validate Supplier Risk output**

In [0]:
# ============================================================
# Validate Gold Supplier Risk output
# ============================================================

supplier_risk_output_count = (
    supplier_risk_gold_df.count()
)


supplier_risk_duplicate_count = (
    supplier_risk_gold_df

    .groupBy(
        "SupplierID",
        "PredictionDate"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


supplier_risk_missing_key_count = (
    supplier_risk_gold_df

    .filter(
        F.col(
            "SupplierKey"
        ).isNull()
    )

    .count()
)


supplier_risk_invalid_score_count = (
    supplier_risk_gold_df

    .filter(
        F.col(
            "SupplierRiskScore"
        ).isNull()
        |
        (
            F.col(
                "SupplierRiskScore"
            )
            < 0
        )
        |
        (
            F.col(
                "SupplierRiskScore"
            )
            > 100
        )
    )

    .count()
)


supplier_risk_invalid_flag_count = (
    supplier_risk_gold_df

    .filter(
        ~F.col(
            "PredictedHighRiskFlag"
        )
        .isin(
            0,
            1
        )
    )

    .count()
)


supplier_risk_date_mismatch_count = (
    supplier_risk_gold_df

    .filter(
        F.col(
            "PredictionDate"
        )
        !=
        F.lit(
            PREDICTION_DATE
        )
    )

    .count()
)


print(
    "Supplier Risk output rows:",
    f"{supplier_risk_output_count:,}"
)

print(
    "Duplicate grain:",
    supplier_risk_duplicate_count
)

print(
    "Missing SupplierKey:",
    supplier_risk_missing_key_count
)

print(
    "Invalid risk scores:",
    supplier_risk_invalid_score_count
)

print(
    "Invalid risk flags:",
    supplier_risk_invalid_flag_count
)

Supplier Risk output rows: 356
Duplicate grain: 0
Missing SupplierKey: 0
Invalid risk scores: 0
Invalid risk flags: 0


**Validate Pricing Anomaly output**

In [0]:
# ============================================================
# Validate Gold Pricing Anomaly output
# ============================================================

pricing_output_count = (
    pricing_anomaly_gold_df.count()
)


pricing_duplicate_count = (
    pricing_anomaly_gold_df

    .groupBy(
        "POItemID",
        "PredictionDate"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


pricing_missing_fact_key_count = (
    pricing_anomaly_gold_df

    .filter(
        F.col(
            "PurchaseOrderFactKey"
        ).isNull()
    )

    .count()
)


pricing_missing_dimension_key_count = (
    pricing_anomaly_gold_df

    .filter(
        F.col(
            "SupplierKey"
        ).isNull()
        |
        F.col(
            "MaterialKey"
        ).isNull()
        |
        F.col(
            "CategoryKey"
        ).isNull()
    )

    .count()
)


pricing_invalid_score_count = (
    pricing_anomaly_gold_df

    .filter(
        F.col(
            "PricingAnomalyScore"
        ).isNull()
        |
        (
            F.col(
                "PricingAnomalyScore"
            )
            < 0
        )
        |
        (
            F.col(
                "PricingAnomalyScore"
            )
            > 100
        )
    )

    .count()
)


pricing_invalid_flag_count = (
    pricing_anomaly_gold_df

    .filter(
        ~F.col(
            "PricingAnomalyFlag"
        )
        .isin(
            0,
            1
        )
    )

    .count()
)


pricing_date_mismatch_count = (
    pricing_anomaly_gold_df

    .filter(
        F.col(
            "PredictionDate"
        )
        !=
        F.lit(
            PREDICTION_DATE
        )
    )

    .count()
)


print(
    "Pricing output rows:",
    f"{pricing_output_count:,}"
)

print(
    "Duplicate grain:",
    pricing_duplicate_count
)

print(
    "Missing PurchaseOrderFactKey:",
    pricing_missing_fact_key_count
)

print(
    "Missing dimensional keys:",
    pricing_missing_dimension_key_count
)

print(
    "Invalid anomaly scores:",
    pricing_invalid_score_count
)

Pricing output rows: 21,752
Duplicate grain: 0
Missing PurchaseOrderFactKey: 0
Missing dimensional keys: 0
Invalid anomaly scores: 0


**Validate Savings Opportunity output**

In [0]:
# ============================================================
# Validate Gold Savings Opportunity output
# ============================================================

savings_output_count = (
    savings_opportunity_gold_df.count()
)


savings_duplicate_count = (
    savings_opportunity_gold_df

    .groupBy(
        "SupplierID",
        "CategoryID",
        "PredictionDate"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


savings_missing_key_count = (
    savings_opportunity_gold_df

    .filter(
        F.col(
            "SupplierKey"
        ).isNull()
        |
        F.col(
            "CategoryKey"
        ).isNull()
    )

    .count()
)


savings_negative_count = (
    savings_opportunity_gold_df

    .filter(
        F.col(
            "PotentialAnnualSavingsEUR"
        )
        < 0
    )

    .count()
)


savings_invalid_priority_count = (
    savings_opportunity_gold_df

    .filter(
        F.col(
            "NegotiationPriorityScore"
        ).isNull()
        |
        (
            F.col(
                "NegotiationPriorityScore"
            )
            < 0
        )
        |
        (
            F.col(
                "NegotiationPriorityScore"
            )
            > 100
        )
    )

    .count()
)


savings_invalid_rank_count = (
    savings_opportunity_gold_df

    .filter(
        F.col(
            "SavingsOpportunityRank"
        ).isNull()
        |
        (
            F.col(
                "SavingsOpportunityRank"
            )
            <= 0
        )
    )

    .count()
)


savings_date_mismatch_count = (
    savings_opportunity_gold_df

    .filter(
        F.col(
            "PredictionDate"
        )
        !=
        F.lit(
            PREDICTION_DATE
        )
    )

    .count()
)


print(
    "Savings output rows:",
    f"{savings_output_count:,}"
)

print(
    "Duplicate grain:",
    savings_duplicate_count
)

print(
    "Missing dimensional keys:",
    savings_missing_key_count
)

print(
    "Negative savings:",
    savings_negative_count
)

print(
    "Invalid priority scores:",
    savings_invalid_priority_count
)

print(
    "Invalid ranks:",
    savings_invalid_rank_count
)

Savings output rows: 983
Duplicate grain: 0
Missing dimensional keys: 0
Negative savings: 0
Invalid priority scores: 0
Invalid ranks: 0


**Cross-product ML integrity checks**

In [0]:
# ============================================================
# Cross-product ML integrity checks
# ============================================================

# ------------------------------------------------------------
# Every supplier used by DB_06 should exist in DB_03 risk
# predictions because DB_06 reported 100% risk coverage.
# ------------------------------------------------------------

savings_supplier_without_risk_count = (
    savings_opportunity_gold_df

    .select(
        "SupplierID"
    )

    .distinct()

    .join(
        supplier_risk_gold_df

        .select(
            "SupplierID"
        )

        .distinct(),

        on="SupplierID",

        how="left_anti"
    )

    .count()
)


# ------------------------------------------------------------
# Ensure both prediction products actually contain positive
# signals rather than degenerate all-zero results.
# ------------------------------------------------------------

high_risk_supplier_count = (
    supplier_risk_gold_df

    .filter(
        F.col(
            "PredictedHighRiskFlag"
        )
        == 1
    )

    .count()
)


pricing_anomaly_flagged_count = (
    pricing_anomaly_gold_df

    .filter(
        F.col(
            "PricingAnomalyFlag"
        )
        == 1
    )

    .count()
)


positive_savings_opportunity_count = (
    savings_opportunity_gold_df

    .filter(
        F.col(
            "PotentialAnnualSavingsEUR"
        )
        > 0
    )

    .count()
)


print(
    "Savings suppliers without risk prediction:",
    savings_supplier_without_risk_count
)

print(
    "High-risk suppliers:",
    f"{high_risk_supplier_count:,}"
)

print(
    "Pricing anomalies:",
    f"{pricing_anomaly_flagged_count:,}"
)

print(
    "Positive savings opportunities:",
    f"{positive_savings_opportunity_count:,}"
)

Savings suppliers without risk prediction: 0
High-risk suppliers: 204
Pricing anomalies: 1,216
Positive savings opportunities: 955


**Final DB_07 pre-write quality gate**

In [0]:
# ============================================================
# DB_07 pre-write quality gate
# ============================================================

quality_checks = [

    # ========================================================
    # Supplier Risk
    # ========================================================

    (
        "Supplier Risk row count preserved",
        supplier_risk_output_count
        ==
        supplier_risk_source_count
    ),

    (
        "Supplier Risk grain is unique",
        supplier_risk_duplicate_count
        == 0
    ),

    (
        "Supplier Risk SupplierKey coverage is complete",
        supplier_risk_missing_key_count
        == 0
    ),

    (
        "Supplier Risk scores are valid",
        supplier_risk_invalid_score_count
        == 0
    ),

    (
        "Supplier Risk flags are valid",
        supplier_risk_invalid_flag_count
        == 0
    ),

    (
        "Supplier Risk prediction date is correct",
        supplier_risk_date_mismatch_count
        == 0
    ),


    # ========================================================
    # Pricing Anomaly
    # ========================================================

    (
        "Pricing Anomaly row count preserved",
        pricing_output_count
        ==
        pricing_anomaly_source_count
    ),

    (
        "Pricing Anomaly grain is unique",
        pricing_duplicate_count
        == 0
    ),

    (
        "Pricing Anomaly PO fact coverage is complete",
        pricing_missing_fact_key_count
        == 0
    ),

    (
        "Pricing Anomaly dimensional coverage is complete",
        pricing_missing_dimension_key_count
        == 0
    ),

    (
        "Pricing Anomaly scores are valid",
        pricing_invalid_score_count
        == 0
    ),

    (
        "Pricing Anomaly flags are valid",
        pricing_invalid_flag_count
        == 0
    ),

    (
        "Pricing Anomaly prediction date is correct",
        pricing_date_mismatch_count
        == 0
    ),


    # ========================================================
    # Savings Opportunity
    # ========================================================

    (
        "Savings Opportunity row count preserved",
        savings_output_count
        ==
        savings_opportunity_source_count
    ),

    (
        "Savings Opportunity grain is unique",
        savings_duplicate_count
        == 0
    ),

    (
        "Savings Opportunity dimensional coverage is complete",
        savings_missing_key_count
        == 0
    ),

    (
        "Savings Opportunity values are non-negative",
        savings_negative_count
        == 0
    ),

    (
        "Savings Opportunity priority scores are valid",
        savings_invalid_priority_count
        == 0
    ),

    (
        "Savings Opportunity ranks are valid",
        savings_invalid_rank_count
        == 0
    ),

    (
        "Savings Opportunity prediction date is correct",
        savings_date_mismatch_count
        == 0
    ),


    # ========================================================
    # Cross-product
    # ========================================================

    (
        "All savings suppliers have Supplier Risk predictions",
        savings_supplier_without_risk_count
        == 0
    ),

    (
        "Supplier Risk output contains high-risk suppliers",
        high_risk_supplier_count
        > 0
    ),

    (
        "Pricing Anomaly output contains anomalies",
        pricing_anomaly_flagged_count
        > 0
    ),

    (
        "Savings output contains positive opportunities",
        positive_savings_opportunity_count
        > 0
    )
]


failed_checks = []


for (
    check_name,
    passed
) in quality_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_checks.append(
            check_name
        )


if failed_checks:

    raise ValueError(
        "DB_07 PRE-WRITE QUALITY GATE FAILED: "
        +
        "; ".join(
            failed_checks
        )
    )


print(
    "\nDB_07 PRE-WRITE QUALITY GATE PASSED."
)

PASS | Supplier Risk row count preserved
PASS | Supplier Risk grain is unique
PASS | Supplier Risk SupplierKey coverage is complete
PASS | Supplier Risk scores are valid
PASS | Supplier Risk flags are valid
PASS | Supplier Risk prediction date is correct
PASS | Pricing Anomaly row count preserved
PASS | Pricing Anomaly grain is unique
PASS | Pricing Anomaly PO fact coverage is complete
PASS | Pricing Anomaly dimensional coverage is complete
PASS | Pricing Anomaly scores are valid
PASS | Pricing Anomaly flags are valid
PASS | Pricing Anomaly prediction date is correct
PASS | Savings Opportunity row count preserved
PASS | Savings Opportunity grain is unique
PASS | Savings Opportunity dimensional coverage is complete
PASS | Savings Opportunity values are non-negative
PASS | Savings Opportunity priority scores are valid
PASS | Savings Opportunity ranks are valid
PASS | Savings Opportunity prediction date is correct
PASS | All savings suppliers have Supplier Risk predictions
PASS | Supplier

**Preview final Gold outputs**

In [0]:
# ============================================================
# Preview governed Gold ML outputs
# ============================================================

print(
    "SUPPLIER RISK"
)

display(
    supplier_risk_gold_df

    .select(
        "SupplierRiskPredictionKey",
        "PredictionDate",
        "SupplierKey",
        "SupplierID",
        "SupplierRiskScore",
        "PredictedHighRiskFlag",
        "ModelName",
        "ModelRunID"
    )

    .orderBy(
        F.desc(
            "SupplierRiskScore"
        )
    )

    .limit(
        20
    )
)


print(
    "PRICING ANOMALIES"
)

display(
    pricing_anomaly_gold_df

    .select(
        "PricingAnomalyPredictionKey",
        "PredictionDate",
        "PurchaseOrderFactKey",
        "POItemID",
        "SupplierKey",
        "MaterialKey",
        "CategoryKey",
        "PricingAnomalyScore",
        "PricingAnomalyFlag",
        "LineAmountEUR",
        "ModelRunID"
    )

    .orderBy(
        F.desc(
            "PricingAnomalyScore"
        )
    )

    .limit(
        20
    )
)


print(
    "SAVINGS OPPORTUNITIES"
)

display(
    savings_opportunity_gold_df

    .select(
        "SavingsOpportunityKey",
        "PredictionDate",
        "SupplierKey",
        "CategoryKey",
        "SupplierID",
        "CategoryID",
        "PotentialAnnualSavingsEUR",
        "NegotiationPriorityScore",
        "NegotiationPriority",
        "SavingsOpportunityRank",
        "ActionableOpportunityFlag"
    )

    .orderBy(
        "SavingsOpportunityRank"
    )

    .limit(
        20
    )
)

SUPPLIER RISK


SupplierRiskPredictionKey,PredictionDate,SupplierKey,SupplierID,SupplierRiskScore,PredictedHighRiskFlag,ModelName,ModelRunID
-114102008740749572,2026-07-31,8813230279933682197,SUP000464,73.71794886390036,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
-5593677719242316613,2026-07-31,-190871052619804443,SUP000191,71.67990250713547,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
6196585204736083859,2026-07-31,2437896912816287331,SUP000102,70.48807107500042,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
-2385161669839471261,2026-07-31,2912409200759669053,SUP000124,70.39585981736447,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
7742104902457964189,2026-07-31,-832554515012663440,SUP000089,70.19726811629954,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
-3932783666890149861,2026-07-31,4104675860832402021,SUP000365,69.86758084628867,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
6417966625739766816,2026-07-31,-110556251672362946,SUP000066,67.78798885159526,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
-287561507653823702,2026-07-31,-1006898620634489255,SUP000258,66.88225963676844,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
5317241401256576119,2026-07-31,4470634542741545506,SUP000178,66.830608018979,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb
862237392735289643,2026-07-31,-2086092482758829123,SUP000259,66.27313137271264,1,RandomForest,b4e4b6964d364b01b117bf86a278f5eb


PRICING ANOMALIES


PricingAnomalyPredictionKey,PredictionDate,PurchaseOrderFactKey,POItemID,SupplierKey,MaterialKey,CategoryKey,PricingAnomalyScore,PricingAnomalyFlag,LineAmountEUR,ModelRunID
-1123553303184891297,2026-07-31,-7035934881797715411,4500002893-00010,7748833086528146996,550035336782325926,6257374930383650811,99.98072707123757,1,264060.7,fdb24ff14d3648bd8e611c6e4792712f
-3030447950110043146,2026-07-31,5972005224589842738,4500000166-00020,7252494749716292028,-5525400466380447165,-3450456260070826903,99.98072707123757,1,51511.89,fdb24ff14d3648bd8e611c6e4792712f
8769816462105205849,2026-07-31,8123471422656042547,4500005958-00010,95111313499456884,1003514827494766228,7572492593648376902,99.97831795514226,1,18721.04,fdb24ff14d3648bd8e611c6e4792712f
-3870287736794415118,2026-07-31,4730710450955064223,4500017328-00010,651412671427293595,-4955890547807597245,-4930974018403207233,99.97831795514226,1,278335.2,fdb24ff14d3648bd8e611c6e4792712f
2731240413036315031,2026-07-31,-3001353528690976123,4500001357-00010,-119821279430463360,5891033991531230573,6313148846500893212,99.97831795514226,1,55182.82,fdb24ff14d3648bd8e611c6e4792712f
6108579381941839886,2026-07-31,2112234320419528393,4500017981-00020,-5115183615864475396,-2066719677703530900,5971194554969453465,99.97831795514226,1,19588.65,fdb24ff14d3648bd8e611c6e4792712f
-6925543155433689812,2026-07-31,-8620137751393230974,4500009917-00070,7913688676740251354,-1068463161614647926,-597818977582192105,99.96145414247512,1,182820.3,fdb24ff14d3648bd8e611c6e4792712f
308307955915462628,2026-07-31,-5193195103334575236,4500016654-00050,7679397451888087664,1511085368158174722,-256265994456941412,99.95663591028452,1,224301.23,fdb24ff14d3648bd8e611c6e4792712f
6929811999625051211,2026-07-31,-5030548902504921354,4500006893-00020,2460006663840093388,1773614982265712287,-3450491637008718913,99.95422679418921,1,205838.9,fdb24ff14d3648bd8e611c6e4792712f
-5578231756285100854,2026-07-31,5047612834400755281,4500009257-00010,-5491926215135380455,3233188606929487393,5971194554969453465,99.95422679418921,1,9990.05,fdb24ff14d3648bd8e611c6e4792712f


SAVINGS OPPORTUNITIES


SavingsOpportunityKey,PredictionDate,SupplierKey,CategoryKey,SupplierID,CategoryID,PotentialAnnualSavingsEUR,NegotiationPriorityScore,NegotiationPriority,SavingsOpportunityRank,ActionableOpportunityFlag
-775048150321282686,2026-07-31,-7851306340512689895,2341845002042429815,SUP000315,CAT010,6930149.86,83.74,CRITICAL,1,1
-7527469101556416958,2026-07-31,-1536108344554802235,-597818977582192105,SUP000242,CAT017,6371084.96,83.54,CRITICAL,2,1
7764030583065896960,2026-07-31,69229922754118892,2341845002042429815,SUP000243,CAT010,4632023.58,81.94,CRITICAL,3,1
-2106114475754125817,2026-07-31,7237942267637074804,2341845002042429815,SUP000295,CAT010,4309287.94,85.6,CRITICAL,4,1
-4698614140335004336,2026-07-31,8460367937439470763,2080465399257655375,SUP000353,CAT009,3758809.71,83.72,CRITICAL,5,1
-6384142817880556104,2026-07-31,4616299021937063669,2341845002042429815,SUP000302,CAT010,3111837.01,84.9,CRITICAL,6,1
6780322563810793751,2026-07-31,-7712797047832536903,2080465399257655375,SUP000419,CAT009,2988682.0,81.93,CRITICAL,7,1
-6240827571889850114,2026-07-31,-7214150169675918194,2080465399257655375,SUP000404,CAT009,2953563.09,79.8,CRITICAL,8,1
-2040002629784374865,2026-07-31,-7214150169675918194,2341845002042429815,SUP000404,CAT010,2591078.32,80.23,CRITICAL,9,1
1919144581614267108,2026-07-31,-7781800243047157220,2341845002042429815,SUP000310,CAT010,1953903.68,81.21,CRITICAL,10,1


**Write validated outputs into physical Gold**

In [0]:
# ============================================================
# Promote validated ML outputs into Fabric Gold
# ============================================================

write_prediction_snapshot(
    supplier_risk_gold_df,
    ML_SUPPLIER_RISK_TARGET_PATH
)


write_prediction_snapshot(
    pricing_anomaly_gold_df,
    ML_PRICING_ANOMALY_TARGET_PATH
)


write_prediction_snapshot(
    savings_opportunity_gold_df,
    ML_SAVINGS_OPPORTUNITY_TARGET_PATH
)


print(
    "\nDB_07 Gold ML promotion completed."
)

**Reload Gold ML tables**

In [0]:
# ============================================================
# Reload persisted Gold ML outputs
# ============================================================

persisted_supplier_risk_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        ML_SUPPLIER_RISK_TARGET_PATH
    )
)


persisted_pricing_anomaly_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        ML_PRICING_ANOMALY_TARGET_PATH
    )
)


persisted_savings_opportunity_df = (
    spark.read
    .format(
        "delta"
    )
    .load(
        ML_SAVINGS_OPPORTUNITY_TARGET_PATH
    )
)


# ------------------------------------------------------------
# Validate only current prediction snapshot
# ------------------------------------------------------------

persisted_supplier_risk_snapshot_df = (
    persisted_supplier_risk_df

    .filter(
        F.col(
            "PredictionDate"
        )
        ==
        F.lit(
            PREDICTION_DATE
        )
    )
)


persisted_pricing_snapshot_df = (
    persisted_pricing_anomaly_df

    .filter(
        F.col(
            "PredictionDate"
        )
        ==
        F.lit(
            PREDICTION_DATE
        )
    )
)


persisted_savings_snapshot_df = (
    persisted_savings_opportunity_df

    .filter(
        F.col(
            "PredictionDate"
        )
        ==
        F.lit(
            PREDICTION_DATE
        )
    )
)


persisted_supplier_risk_count = (
    persisted_supplier_risk_snapshot_df.count()
)


persisted_pricing_count = (
    persisted_pricing_snapshot_df.count()
)


persisted_savings_count = (
    persisted_savings_snapshot_df.count()
)


print(
    "Persisted Supplier Risk snapshot:",
    f"{persisted_supplier_risk_count:,}"
)

print(
    "Persisted Pricing Anomaly snapshot:",
    f"{persisted_pricing_count:,}"
)

print(
    "Persisted Savings Opportunity snapshot:",
    f"{persisted_savings_count:,}"
)

Persisted Supplier Risk snapshot: 356
Persisted Pricing Anomaly snapshot: 21,752
Persisted Savings Opportunity snapshot: 983


**Monetary and classification reconciliation**

In [0]:
# ============================================================
# Reconcile persisted Gold ML snapshots with source outputs
# ============================================================

# ------------------------------------------------------------
# Supplier risk
# ------------------------------------------------------------

source_high_risk_count = (
    supplier_risk_gold_df

    .filter(
        F.col(
            "PredictedHighRiskFlag"
        )
        == 1
    )

    .count()
)


persisted_high_risk_count = (
    persisted_supplier_risk_snapshot_df

    .filter(
        F.col(
            "PredictedHighRiskFlag"
        )
        == 1
    )

    .count()
)


# ------------------------------------------------------------
# Pricing anomaly
# ------------------------------------------------------------

source_anomaly_count = (
    pricing_anomaly_gold_df

    .filter(
        F.col(
            "PricingAnomalyFlag"
        )
        == 1
    )

    .count()
)


persisted_anomaly_count = (
    persisted_pricing_snapshot_df

    .filter(
        F.col(
            "PricingAnomalyFlag"
        )
        == 1
    )

    .count()
)


# ------------------------------------------------------------
# Savings opportunity
# ------------------------------------------------------------

source_total_savings = (
    savings_opportunity_gold_df

    .agg(
        F.sum(
            "PotentialAnnualSavingsEUR"
        )
        .alias(
            "PotentialAnnualSavingsEUR"
        )
    )

    .first()[
        "PotentialAnnualSavingsEUR"
    ]
)


persisted_total_savings = (
    persisted_savings_snapshot_df

    .agg(
        F.sum(
            "PotentialAnnualSavingsEUR"
        )
        .alias(
            "PotentialAnnualSavingsEUR"
        )
    )

    .first()[
        "PotentialAnnualSavingsEUR"
    ]
)


savings_difference = (
    abs(
        float(
            source_total_savings
        )
        -
        float(
            persisted_total_savings
        )
    )
)


print(
    "High-risk supplier difference:",
    (
        source_high_risk_count
        -
        persisted_high_risk_count
    )
)

print(
    "Pricing anomaly difference:",
    (
        source_anomaly_count
        -
        persisted_anomaly_count
    )
)

print(
    "Savings EUR difference:",
    round(
        savings_difference,
        2
    )
)

High-risk supplier difference: 0
Pricing anomaly difference: 0
Savings EUR difference: 0.0


**Post-write persistance gate**

In [0]:
# ============================================================
# DB_07 persistence quality gate
# ============================================================

persistence_checks = [

    (
        "Supplier Risk persisted row count matches source",
        persisted_supplier_risk_count
        ==
        supplier_risk_output_count
    ),

    (
        "Pricing Anomaly persisted row count matches source",
        persisted_pricing_count
        ==
        pricing_output_count
    ),

    (
        "Savings Opportunity persisted row count matches source",
        persisted_savings_count
        ==
        savings_output_count
    ),

    (
        "High-risk classification reconciles",
        source_high_risk_count
        ==
        persisted_high_risk_count
    ),

    (
        "Pricing anomaly classification reconciles",
        source_anomaly_count
        ==
        persisted_anomaly_count
    ),

    (
        "Potential Annual Savings reconciles",
        savings_difference
        <= 0.02
    )
]


failed_persistence_checks = []


for (
    check_name,
    passed
) in persistence_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_persistence_checks.append(
            check_name
        )


if failed_persistence_checks:

    raise ValueError(
        "DB_07 PERSISTENCE QUALITY GATE FAILED: "
        +
        "; ".join(
            failed_persistence_checks
        )
    )


print(
    "\nDB_07 PERSISTENCE QUALITY GATE PASSED."
)

PASS | Supplier Risk persisted row count matches source
PASS | Pricing Anomaly persisted row count matches source
PASS | Savings Opportunity persisted row count matches source
PASS | High-risk classification reconciles
PASS | Pricing anomaly classification reconciles
PASS | Potential Annual Savings reconciles

DB_07 PERSISTENCE QUALITY GATE PASSED.


**Build Gold ML promotion monitoring record**

In [0]:
# ============================================================
# Build ML Gold promotion monitoring records
# ============================================================

monitoring_rows = [

    (
        "ml_supplier_risk_prediction",

        SUPPLIER_RISK_SOURCE_PATH,

        ML_SUPPLIER_RISK_TARGET_PATH,

        int(
            supplier_risk_source_count
        ),

        int(
            persisted_supplier_risk_count
        ),

        int(
            supplier_risk_duplicate_count
        ),

        int(
            supplier_risk_missing_key_count
        ),

        int(
            supplier_risk_invalid_score_count
        ),

        "PASS"
    ),

    (
        "ml_pricing_anomaly_prediction",

        PRICING_ANOMALY_SOURCE_PATH,

        ML_PRICING_ANOMALY_TARGET_PATH,

        int(
            pricing_anomaly_source_count
        ),

        int(
            persisted_pricing_count
        ),

        int(
            pricing_duplicate_count
        ),

        int(
            pricing_missing_fact_key_count
            +
            pricing_missing_dimension_key_count
        ),

        int(
            pricing_invalid_score_count
        ),

        "PASS"
    ),

    (
        "ml_savings_opportunity",

        SAVINGS_OPPORTUNITY_SOURCE_PATH,

        ML_SAVINGS_OPPORTUNITY_TARGET_PATH,

        int(
            savings_opportunity_source_count
        ),

        int(
            persisted_savings_count
        ),

        int(
            savings_duplicate_count
        ),

        int(
            savings_missing_key_count
        ),

        int(
            savings_negative_count
            +
            savings_invalid_priority_count
        ),

        "PASS"
    )
]


monitoring_schema = T.StructType([

    T.StructField(
        "TableName",
        T.StringType(),
        False
    ),

    T.StructField(
        "SourcePath",
        T.StringType(),
        False
    ),

    T.StructField(
        "TargetPath",
        T.StringType(),
        False
    ),

    T.StructField(
        "SourceRowCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "PersistedRowCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "DuplicateRecordCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "OrphanRecordCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "InvalidMetricCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "ValidationStatus",
        T.StringType(),
        False
    )
])


monitoring_ml_gold_promotion_df = (
    spark.createDataFrame(
        monitoring_rows,
        monitoring_schema
    )

    .withColumn(
        "PredictionDate",
        F.lit(
            PREDICTION_DATE
        )
        .cast(
            "date"
        )
    )

    .withColumn(
        "PromotionBatchID",
        F.lit(
            PROMOTION_BATCH_ID
        )
    )

    .withColumn(
        "ExecutionTimestampUTC",
        F.lit(
            PROMOTION_TIMESTAMP_UTC
        )
        .cast(
            "timestamp"
        )
    )
)


display(
    monitoring_ml_gold_promotion_df
)

TableName,SourcePath,TargetPath,SourceRowCount,PersistedRowCount,DuplicateRecordCount,OrphanRecordCount,InvalidMetricCount,ValidationStatus,PredictionDate,PromotionBatchID,ExecutionTimestampUTC
ml_supplier_risk_prediction,abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/predictions_2026,abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/ml_supplier_risk_prediction,356,356,0,0,0,PASS,2026-07-31,a222cd41-aa4e-4afa-848d-ccb194f35f4a,2026-08-13T15:23:04.248548Z
ml_pricing_anomaly_prediction,abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/pricing_anomaly/scoring_predictions,abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/ml_pricing_anomaly_prediction,21752,21752,0,0,0,PASS,2026-07-31,a222cd41-aa4e-4afa-848d-ccb194f35f4a,2026-08-13T15:23:04.248548Z
ml_savings_opportunity,abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/savings_opportunity/opportunities_2026,abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/ml_savings_opportunity,983,983,0,0,0,PASS,2026-07-31,a222cd41-aa4e-4afa-848d-ccb194f35f4a,2026-08-13T15:23:04.248548Z


**Persist promotion monitoring**

In [0]:
# ============================================================
# Persist DB_07 promotion monitoring history
# ============================================================

if DeltaTable.isDeltaTable(
    spark,
    ML_PROMOTION_MONITORING_PATH
):

    (
        monitoring_ml_gold_promotion_df

        .write

        .format(
            "delta"
        )

        .mode(
            "append"
        )

        .save(
            ML_PROMOTION_MONITORING_PATH
        )
    )


else:

    (
        monitoring_ml_gold_promotion_df

        .write

        .format(
            "delta"
        )

        .mode(
            "overwrite"
        )

        .option(
            "overwriteSchema",
            "true"
        )

        .save(
            ML_PROMOTION_MONITORING_PATH
        )
    )


print(
    "ML Gold promotion monitoring persisted."
)

ML Gold promotion monitoring persisted.


**Final Gold ML portfolio summary**

In [0]:
# ============================================================
# Final DB_07 Gold ML portfolio summary
# ============================================================

final_summary_schema = T.StructType([

    T.StructField(
        "MLDataProduct",
        T.StringType(),
        False
    ),

    T.StructField(
        "RowCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "PositiveSignalCount",
        T.LongType(),
        False
    ),

    T.StructField(
        "PotentialAnnualSavingsEUR",
        T.DoubleType(),
        True
    ),

    T.StructField(
        "Notes",
        T.StringType(),
        True
    )
])


final_summary_rows = [

    (
        "Supplier Risk Prediction",

        int(
            persisted_supplier_risk_count
        ),

        int(
            high_risk_supplier_count
        ),

        None,

        (
            "Supplier-level predicted risk "
            "for the current prediction snapshot"
        )
    ),

    (
        "Pricing Anomaly Prediction",

        int(
            persisted_pricing_count
        ),

        int(
            pricing_anomaly_flagged_count
        ),

        None,

        (
            "PO-item pricing anomalies identified "
            "by Isolation Forest"
        )
    ),

    (
        "Savings Opportunity",

        int(
            persisted_savings_count
        ),

        int(
            positive_savings_opportunity_count
        ),

        float(
            persisted_total_savings
        ),

        (
            "Supplier-category prescriptive "
            "savings opportunities"
        )
    )
]


final_summary_df = (
    spark.createDataFrame(
        final_summary_rows,
        schema=final_summary_schema
    )
)


display(
    final_summary_df
)


print(
    "Physical Gold ML tables:"
)

print(
    "1. ml_supplier_risk_prediction"
)

print(
    "2. ml_pricing_anomaly_prediction"
)

print(
    "3. ml_savings_opportunity"
)

print(
    "4. monitoring_ml_gold_promotion_results"
)


print(
    "\nDB_07 SCORE AND WRITE ML OUTPUTS PASSED."
)

MLDataProduct,RowCount,PositiveSignalCount,PotentialAnnualSavingsEUR,Notes
Supplier Risk Prediction,356,204,null,Supplier-level predicted risk for the current prediction snapshot
Pricing Anomaly Prediction,21752,1216,null,PO-item pricing anomalies identified by Isolation Forest
Savings Opportunity,983,955,9.754746873000005E7,Supplier-category prescriptive savings opportunities


Physical Gold ML tables:
1. ml_supplier_risk_prediction
2. ml_pricing_anomaly_prediction
3. ml_savings_opportunity
4. monitoring_ml_gold_promotion_results

DB_07 SCORE AND WRITE ML OUTPUTS PASSED.
